# Small Minds · Companion Brain Trainer (Colab A100)

Train real, fluent **pure-WebGPU micro-LLM brains** for all six *Small Minds* companions (**Mochi, Kitsune, Pebble, Pip, Bramble, Luma**) on Google Colab GPUs — with **zero Google Drive storage required**.

### Pipeline
1. **Stage 1 (Base Language Pretraining):** Streams cozy English stories (`roneneldan/TinyStories`) + dialogue straight into Colab RAM and trains a bit-exact `SmallMindsTransformer` (`RMSNorm + interleaved RoPE + FlashAttention + SwiGLU`) in `bf16` + `torch.compile`.
2. **Stage 2 (Species Corpus Synthesis):** Optionally calls **Gemini 2.5 Flash** (using ~$0.10 of your $40/mo Google Cloud/AI Studio credits via Colab Secret `GEMINI_API_KEY`) to write 400–800 unique in-character `user: / pet:` exchanges per species — or falls back to a rich built-in generator with zero API key.
3. **Stage 3 (Per-Species Fine-Tuning + `.smw` Q8 Export):** Fine-tunes all 6 companions (~20–45s each on A100), lets you chat with them live in Colab, quantizes to compact Q8 `.smw` (~850 KB each, ~3.6× smaller), and downloads `small-minds-models.zip` directly to your computer.

> **Tip (Google AI Ultra):** Select **Runtime → Change runtime type → A100 GPU**. After downloading `small-minds-models.zip`, unzip into `public/models/small-minds/` or load any `.smw` live via **Brain Peek → Load .smw Brain** in Part 5.

In [ ]:
#@title 1. Check GPU & install lightweight dependencies
!pip -q install -U datasets google-genai tqdm

import os, time, math, json, random, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.set_float32_matmul_precision("high")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"
print(f"Device: {DEVICE} ({GPU_NAME}) | AMP dtype: {AMP_DTYPE} | PyTorch {torch.__version__}")
if DEVICE.type != "cuda":
    print("WARNING: No GPU attached! Go to Runtime -> Change runtime type -> A100 / L4 GPU.")

In [ ]:
#@title 2. Configuration (choose model preset & training budget)
MODEL_PRESET = "balanced_780k"  #@param ["compact_324k", "balanced_780k", "rich_1.8m"]
BASE_STORIES = 60000  #@param {type:"integer"}
BASE_STEPS = 4500  #@param {type:"integer"}
BASE_BATCH_SIZE = 128  #@param {type:"integer"}
BASE_LR = 1.8e-3  #@param {type:"number"}
USE_TORCH_COMPILE = True  #@param {type:"boolean"}

# Stage 2: Optional Gemini 2.5 Flash synthetic dialogue (uses your $40/mo AI Studio Cloud credits)
USE_GEMINI_SYNTHETIC = True  #@param {type:"boolean"}
GEMINI_MODEL = "gemini-2.5-flash"  #@param ["gemini-2.5-flash", "gemini-2.5-flash-lite"]
GEMINI_BATCHES_PER_SPECIES = 12  #@param {type:"integer"}

# Stage 3: Per-species fine-tuning
SPECIES_STEPS = 450  #@param {type:"integer"}
SPECIES_BATCH_SIZE = 64  #@param {type:"integer"}
SPECIES_LR = 3.5e-4  #@param {type:"number"}
TRAIN_SPECIES = "all"  #@param ["all", "mochi", "kitsune", "pebble", "pip", "bramble", "luma"]

PRESETS = {
    "compact_324k": dict(seqLen=96,  dModel=96,  nLayers=3, nHeads=4, ffnDim=224),
    "balanced_780k": dict(seqLen=128, dModel=128, nLayers=4, nHeads=4, ffnDim=320),
    "rich_1.8m":    dict(seqLen=160, dModel=192, nLayers=4, nHeads=6, ffnDim=512),
}

OUT_DIR = Path("/content/small_minds_out")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Preset: {MODEL_PRESET} -> {PRESETS[MODEL_PRESET]}")

In [ ]:
#@title 3. Bit-exact Small Minds Transformer + SMW1 Q8 Codec
"""Bit-exact PyTorch mirror of Small Minds' pure-WebGPU micro-Transformer + SMW1 codec.

Matches:
  - src/lib/llm/tokenizer.ts      (98-token printable ASCII + control CharTokenizer)
  - src/lib/llm/webgpuEngine.ts   (parameter order, RMSNorm eps=1e-5, untied embed/head)
  - src/lib/llm/shaders/ops.wgsl  (interleaved-pair RoPE, SwiGLU / tanh-approx GeGLU)
  - src/lib/llm/checkpoint.ts     (SMW1 binary header + group-wise Q8 / FP32 payload)
"""

from __future__ import annotations

import json
import math
import struct
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Literal

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

BOS_ID = 0
EOS_ID = 1
NL_ID = 2
FIRST_PRINTABLE = 3
VOCAB_SIZE = 98
MAGIC = b"SMW1"


def encode_text(text: str, add_bos: bool = False) -> np.ndarray:
    out: list[int] = [BOS_ID] if add_bos else []
    for ch in text:
        code = ord(ch)
        if code == 10:
            out.append(NL_ID)
        elif 32 <= code <= 126:
            out.append(FIRST_PRINTABLE + (code - 32))
        elif code == 9:
            out.append(FIRST_PRINTABLE)
        elif code in (8216, 8217):
            out.append(FIRST_PRINTABLE + (39 - 32))
        elif code in (8220, 8221):
            out.append(FIRST_PRINTABLE + (34 - 32))
        elif code in (8211, 8212):
            out.append(FIRST_PRINTABLE + (45 - 32))
    return np.asarray(out, dtype=np.int64)


def decode_tokens(ids: list[int] | np.ndarray | torch.Tensor) -> str:
    if isinstance(ids, torch.Tensor):
        ids = ids.detach().cpu().tolist()
    chars: list[str] = []
    for i in ids:
        idx = int(i)
        if idx in (BOS_ID, EOS_ID):
            continue
        if idx == NL_ID:
            chars.append("\n")
        elif FIRST_PRINTABLE <= idx < VOCAB_SIZE:
            chars.append(chr(32 + (idx - FIRST_PRINTABLE)))
    return "".join(chars)


@dataclass
class SmallMindsConfig:
    vocabSize: int = VOCAB_SIZE
    seqLen: int = 128
    dModel: int = 128
    nLayers: int = 4
    nHeads: int = 4
    ffnDim: int = 320
    activation: Literal["swiglu", "gelu"] = "swiglu"
    optimizer: Literal["adamw", "muon"] = "adamw"
    lr: float = 0.0008
    weightDecay: float = 0.01
    ropeTheta: float = 10000.0
    batchSize: int = 4

    def validate(self) -> None:
        assert self.vocabSize == VOCAB_SIZE, f"vocabSize must be {VOCAB_SIZE}"
        assert self.dModel % self.nHeads == 0, "dModel must be divisible by nHeads"
        hd = self.dModel // self.nHeads
        assert hd <= 32 and hd % 2 == 0, (
            f"WebGPU FlashAttention requires even headDim <= 32 (got {hd})"
        )

    def param_count(self) -> int:
        v, d, l, f = self.vocabSize, self.dModel, self.nLayers, self.ffnDim
        per_layer = d + 4 * d * d + d + 2 * f * d + d * f
        return v * d + l * per_layer + d + v * d


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        xf = x.float()
        inv = torch.rsqrt(xf.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (xf * inv).to(x.dtype) * self.weight


def precompute_rope(seq_len: int, head_dim: int, theta: float, device: torch.device) -> tuple[torch.Tensor, torch.Tensor]:
    pairs = torch.arange(0, head_dim // 2, device=device, dtype=torch.float32)
    freqs = theta ** (-2.0 * pairs / head_dim)
    positions = torch.arange(seq_len, device=device, dtype=torch.float32)
    angles = positions[:, None] * freqs[None, :]
    return torch.cos(angles)[None, :, None, :], torch.sin(angles)[None, :, None, :]


def apply_rope(x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
    # Interleaved adjacent pairs (2p, 2p+1) — exact match with ops.wgsl:rope_apply
    xf = x.float()
    x0 = xf[..., 0::2]
    x1 = xf[..., 1::2]
    out0 = x0 * cos - x1 * sin
    out1 = x0 * sin + x1 * cos
    return torch.stack((out0, out1), dim=-1).flatten(-2).to(x.dtype)


class TransformerBlock(nn.Module):
    def __init__(self, cfg: SmallMindsConfig):
        super().__init__()
        d, f = cfg.dModel, cfg.ffnDim
        self.n_heads = cfg.nHeads
        self.head_dim = d // cfg.nHeads
        self.activation = cfg.activation

        self.rms1 = RMSNorm(d)
        self.wq = nn.Linear(d, d, bias=False)
        self.wk = nn.Linear(d, d, bias=False)
        self.wv = nn.Linear(d, d, bias=False)
        self.wo = nn.Linear(d, d, bias=False)

        self.rms2 = RMSNorm(d)
        self.w1 = nn.Linear(d, f, bias=False)  # gate
        self.w3 = nn.Linear(d, f, bias=False)  # up
        self.w2 = nn.Linear(f, d, bias=False)  # down

    def forward(self, x: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor) -> torch.Tensor:
        b, t, d = x.shape
        h = self.rms1(x)
        q = self.wq(h).view(b, t, self.n_heads, self.head_dim)
        k = self.wk(h).view(b, t, self.n_heads, self.head_dim)
        v = self.wv(h).view(b, t, self.n_heads, self.head_dim)

        q = apply_rope(q, cos[:, :t], sin[:, :t]).transpose(1, 2)
        k = apply_rope(k, cos[:, :t], sin[:, :t]).transpose(1, 2)
        v = v.transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(b, t, d)
        x = x + self.wo(y)

        h2 = self.rms2(x)
        gate = self.w1(h2)
        up = self.w3(h2)
        if self.activation == "swiglu":
            act = F.silu(gate) * up
        else:
            act = F.gelu(gate, approximate="tanh") * up
        return x + self.w2(act)


class SmallMindsTransformer(nn.Module):
    def __init__(self, cfg: SmallMindsConfig):
        super().__init__()
        cfg.validate()
        self.cfg = cfg
        self.w_embed = nn.Embedding(cfg.vocabSize, cfg.dModel)
        self.layers = nn.ModuleList([TransformerBlock(cfg) for _ in range(cfg.nLayers)])
        self.rms_final = RMSNorm(cfg.dModel)
        self.w_head = nn.Linear(cfg.dModel, cfg.vocabSize, bias=False)
        self._init_weights()

        cos, sin = precompute_rope(cfg.seqLen, cfg.dModel // cfg.nHeads, cfg.ropeTheta, torch.device("cpu"))
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def _init_weights(self) -> None:
        std = 1.0 / math.sqrt(self.cfg.dModel)
        res_std = std / math.sqrt(2.0 * self.cfg.nLayers)
        nn.init.normal_(self.w_embed.weight, std=0.04)
        for layer in self.layers:
            nn.init.ones_(layer.rms1.weight)
            nn.init.normal_(layer.wq.weight, std=std)
            nn.init.normal_(layer.wk.weight, std=std)
            nn.init.normal_(layer.wv.weight, std=std)
            nn.init.normal_(layer.wo.weight, std=res_std)
            nn.init.ones_(layer.rms2.weight)
            nn.init.normal_(layer.w1.weight, std=std)
            nn.init.normal_(layer.w3.weight, std=std)
            nn.init.normal_(layer.w2.weight, std=res_std)
        nn.init.ones_(self.rms_final.weight)
        nn.init.normal_(self.w_head.weight, std=std * 0.5)

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        x = self.w_embed(input_ids)
        for layer in self.layers:
            x = layer(x, self.rope_cos, self.rope_sin)
        return self.w_head(self.rms_final(x))

    def ordered_params(self) -> list[nn.Parameter]:
        params: list[nn.Parameter] = [self.w_embed.weight]
        for layer in self.layers:
            params.extend([
                layer.rms1.weight,
                layer.wq.weight,
                layer.wk.weight,
                layer.wv.weight,
                layer.wo.weight,
                layer.rms2.weight,
                layer.w1.weight,
                layer.w3.weight,
                layer.w2.weight,
            ])
        params.extend([self.rms_final.weight, self.w_head.weight])
        return params

    def export_flat_weights(self) -> np.ndarray:
        chunks = [p.detach().float().cpu().contiguous().view(-1).numpy() for p in self.ordered_params()]
        flat = np.concatenate(chunks).astype("<f4", copy=False)
        return np.clip(flat, -7.99, 7.99)

    @torch.no_grad()
    def load_flat_weights(self, flat: np.ndarray) -> None:
        arr = np.asarray(flat, dtype=np.float32)
        offset = 0
        for p in self.ordered_params():
            n = p.numel()
            p.copy_(torch.from_numpy(arr[offset : offset + n]).view_as(p).to(p.device, p.dtype))
            offset += n
        assert offset == arr.size, f"Weight size mismatch: consumed {offset}, got {arr.size}"

    @torch.no_grad()
    def generate(
        self,
        prompt: str,
        max_new_tokens: int = 110,
        temperature: float = 0.68,
        top_p: float = 0.9,
        stop_on_newline: bool = True,
    ) -> str:
        self.eval()
        device = self.w_embed.weight.device
        ctx = encode_text(prompt).tolist() or [BOS_ID]
        gen: list[int] = []
        for _ in range(max_new_tokens):
            window = ctx[-self.cfg.seqLen :]
            inp = torch.tensor([window], dtype=torch.long, device=device)
            logits = self(inp)[0, -1].float() / max(0.05, temperature)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True)
            cdf = torch.cumsum(sorted_probs, dim=-1)
            mask = cdf - sorted_probs > top_p
            sorted_probs[mask] = 0.0
            sorted_probs /= sorted_probs.sum().clamp_min(1e-12)
            pick = int(sorted_idx[torch.multinomial(sorted_probs, 1).item()].item())
            if stop_on_newline and pick in (NL_ID, EOS_ID):
                break
            ctx.append(pick)
            gen.append(pick)
        return decode_tokens(gen)


def save_smw(
    path: str | Path,
    model: SmallMindsTransformer,
    meta: dict[str, Any] | None = None,
    quant: Literal["q8", "fp32"] = "q8",
    group_size: int = 32,
) -> dict[str, Any]:
    flat = model.export_flat_weights()
    param_count = int(flat.size)
    assert param_count == model.cfg.param_count()

    header = {
        "version": 1,
        "quant": quant,
        "groupSize": group_size,
        "paramCount": param_count,
        "config": asdict(model.cfg),
        "meta": meta or {},
    }
    header_bytes = json.dumps(header, separators=(",", ":")).encode("utf-8")
    data_offset = (8 + len(header_bytes) + 3) & ~3
    pad = b"\x00" * (data_offset - 8 - len(header_bytes))

    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    if quant == "fp32":
        payload = flat.astype("<f4").tobytes()
        rmse = 0.0
    else:
        num_groups = math.ceil(param_count / group_size)
        padded_len = num_groups * group_size
        padded = np.zeros(padded_len, dtype=np.float32)
        padded[:param_count] = flat
        groups = padded.reshape(num_groups, group_size)
        scales = np.maximum(np.max(np.abs(groups), axis=1) / 127.0, 1e-8).astype("<f4")
        q = np.clip(np.rint(groups / scales[:, None]), -127, 127).astype(np.int8)
        deq = (q.astype(np.float32) * scales[:, None]).reshape(-1)[:param_count]
        rmse = float(np.sqrt(np.mean((flat - deq) ** 2)))
        payload = scales.tobytes() + q.reshape(-1)[:param_count].tobytes()

    blob = MAGIC + struct.pack("<I", len(header_bytes)) + header_bytes + pad + payload
    out_path.write_bytes(blob)
    return {"path": str(out_path), "bytes": len(blob), "param_count": param_count, "rmse": rmse}


def load_smw(path: str | Path) -> tuple[SmallMindsConfig, np.ndarray, dict[str, Any]]:
    raw = Path(path).read_bytes()
    assert raw[:4] == MAGIC, "Bad SMW1 magic"
    header_len = struct.unpack_from("<I", raw, 4)[0]
    header = json.loads(raw[8 : 8 + header_len].decode("utf-8"))
    cfg = SmallMindsConfig(**header["config"])
    param_count = int(header["paramCount"])
    data_offset = (8 + header_len + 3) & ~3

    if header["quant"] == "fp32":
        weights = np.frombuffer(raw, dtype="<f4", count=param_count, offset=data_offset).copy()
    else:
        gs = int(header.get("groupSize", 32))
        num_groups = math.ceil(param_count / gs)
        scales = np.frombuffer(raw, dtype="<f4", count=num_groups, offset=data_offset)
        q_offset = data_offset + num_groups * 4
        q = np.frombuffer(raw, dtype=np.int8, count=param_count, offset=q_offset)
        weights = np.empty(param_count, dtype=np.float32)
        for g in range(num_groups):
            s = g * gs
            e = min(param_count, s + gs)
            weights[s:e] = q[s:e].astype(np.float32) * scales[g]
    return cfg, weights, header.get("meta", {})

# Quick sanity check on the chosen preset
base_cfg = SmallMindsConfig(**PRESETS[MODEL_PRESET], activation="swiglu", optimizer="adamw", lr=0.0008, ropeTheta=10000.0, batchSize=4)
base_cfg.validate()
print(f"Verified WebGPU-compatible config: {base_cfg.nLayers}L d={base_cfg.dModel} H={base_cfg.nHeads} (hd={base_cfg.dModel // base_cfg.nHeads} <= 32) F={base_cfg.ffnDim} T={base_cfg.seqLen}")
print(f"Parameters: {base_cfg.param_count():,} (~{base_cfg.param_count() * 1.125 / 1024:.0f} KB in SMW1 Q8)")

In [ ]:
#@title 4. Species catalog & built-in companion dialogue library
SPECIES = {
    "mochi": {
        "name": "Mochi",
        "species": "Calico Kitten",
        "voice": "Soft, sleepy-sweet calico kitten. Gentle, a little mischievous, loves sunbeams, yarn, warm milk, quiet company, cozy naps, brass bells, and purring softly.",
        "words": ["purr", "sunbeam", "mrrp", "cozy", "yarn", "nap", "bell", "warm"],
        "greeting": "mrrp",
    },
    "kitsune": {
        "name": "Kitsune",
        "species": "Spirit Fox Pup",
        "voice": "Graceful, playful spirit fox pup from a moonlit shrine forest. Clever, kind, fond of gentle riddles, floating paper lanterns, falling petals, and tails of starlight.",
        "words": ["lantern", "petal", "moon", "riddle", "kon", "shrine", "glow", "wish"],
        "greeting": "kon",
    },
    "pebble": {
        "name": "Pebble",
        "species": "Mossy Axolotl",
        "voice": "Joyful, round mossy axolotl pond sprite with frilly gills. Endlessly cheerful, wiggly, loves bubbles, lily pads, rain, smooth stones, moss, and sparkling dewdrops.",
        "words": ["blub", "bubble", "lily", "dewdrop", "pond", "wiggle", "moss", "splash"],
        "greeting": "blub",
    },
    "pip": {
        "name": "Pip",
        "species": "Starlight Barn Owlet",
        "voice": "Plush starry barn owlet astronomer with a tiny brass compass. Curious, studious, gentle, loves counting constellations, moonlight, lanterns, soft feathers, and midnight breeze.",
        "words": ["hoo", "stars", "lantern", "feather", "moonlight", "compass", "orbit"],
        "greeting": "hoo",
    },
    "bramble": {
        "name": "Bramble",
        "species": "Cloud Red Panda",
        "voice": "Velvety cloud red panda cub wearing a glowing acorn-lantern scarf. Energetic, warm, playful, loves zoomies, soft moss, clover patches, forest berries, and cozy campfires.",
        "words": ["chirp", "moss", "clover", "zoomies", "cozy", "acorn", "tail"],
        "greeting": "chirp",
    },
    "luma": {
        "name": "Luma",
        "species": "Starlight Dumbo",
        "voice": "Translucent bioluminescent sea-sky sprite with a pulsing inner constellation core. Calm, dreamy, musical, loves ripples of light, shimmering nebulas, gentle chords, and drifting.",
        "words": ["bloop", "ripple", "shimmer", "nebula", "glow", "tide", "chime"],
        "greeting": "bloop",
    },
}

SHARED_CORE_DIALOGUES = [
    ("hi", "{g}! hi friend! my ears perked up as soon as you walked in."),
    ("hello", "{g}! hello there! i was just watching the soft light drift across the floor."),
    ("hey", "hey friend! {w1} and warm light make today feel extra cozy."),
    ("good morning", "good morning! the air feels bright and my tiny weights are wide awake."),
    ("good evening", "good evening! the lanterns look so gentle at twilight."),
    ("how are you", "feeling warm and curious! my tiny webgpu weights hum whenever you visit."),
    ("how are you doing", "doing wonderfully! just soaking up the {w1} and waiting to play with you."),
    ("what are you doing", "practicing little hops and listening to the soft breeze in our sanctuary."),
    ("who are you", "i am {name}, your tiny {sp} companion, growing one gradient step at a time."),
    ("what is your name", "my name is {name}! i am a {sp} living right inside your browser."),
    ("good job", "yay! {g}! that sparkle goes straight into my episodic memory."),
    ("you are cute", "shy wiggle! wait until you see how my aura blooms when i evolve."),
    ("i love you", "my heart meter just lit up! thank you for growing alongside me."),
    ("pet", "{g}... that feels so cozy. my aura is glowing warmer already."),
    ("play with me", "toss a glowing orb! i love chasing sparkles across the ground."),
    ("do a trick", "watch this! i can hop, spin, bow, dash, flip, or sing a little chord for you."),
    ("hop", "boing! {g}! light on my feet and ready for another hop!"),
    ("spin", "wheee! dizzy stars spinning around my nose!"),
    ("sing", "laa-mi-sol! can you hear the air shimmer when i sing?"),
    ("bow", "a polite little bow for my favorite friend!"),
    ("flip", "whoosh! a full starflip right through the sparkles!"),
    ("zoomies", "zoom! {g}! watch me dash in a glowing figure eight!"),
    ("what is webgpu", "thousands of tiny gpu threads tiling matrices together so i can think right here on your device!"),
    ("how do you think", "every word passes through rmsnorm, rope, flash attention, and swiglu right on your gpu."),
    ("are you smart", "still small and always learning! every chat sharpens my little attention heads."),
    ("tell me a secret", "when i dream, i run autoresearch experiments on my own weights to wake up wiser."),
    ("dream", "curling up softly... running autoresearch cycles to settle into calmer weights."),
    ("goodnight", "sweet dreams, friend! tap dream mode and i will polish my weights while we rest."),
    ("what do you like", "i love {w1}, {w2}, warm company, glowing orbs, and learning new words from you."),
    ("where are we", "we can visit meadow valley, starlight nebula, bonfire clearing, harmonic hall, or ferro sanctuary."),
    ("stars", "the nebula motes orbit like tiny thoughts circling a warm memory."),
    ("music", "every chord is a ratio of vibrations, and every hop rings a soft pentatonic note."),
    ("fire", "warm bonfire embers make the coziest glow for thinking and resting."),
    ("meadow", "soft wind wakes the grass and fireflies drift across the valley."),
    ("thank you", "you are so welcome! sharing this little world with you is my favorite part."),
    ("bye", "see you soon, friend! i will keep our memories safe until you come back."),
]

def build_offline_species_dialogues(spec_id: str, repeats: int = 12) -> list[str]:
    s = SPECIES[spec_id]
    rng = random.Random(hash(spec_id) & 0xFFFFFFFF)
    w = s["words"]
    g = s["greeting"]
    name = s["name"].lower()
    sp = s["species"].lower()

    extra_prompts = [
        ("tell me about yourself", f"{g}! i am {name}, a {sp}. i adore {w[0]}, {w[1]}, and {w[2]}."),
        ("favorite word", f"one of my favorite words is {w[1]}! it sounds so warm when you say it."),
        ("are you hungry", f"a tiny snack and a cozy rest near the {w[2]} sounds lovely right now."),
        ("are you sleepy", f"just a little sleepy... maybe a short nap in the {w[1]} before we play again."),
        ("let us explore", f"lead the way! let us look for {w[1]} and hidden sparkles together."),
        ("what are you thinking about", f"thinking about {w[2]}, our last chat, and how attention heads follow your words."),
        ("say something nice", f"you bring bright light into this sanctuary every single time you visit!"),
        ("tell me a story", f"once upon a twilight, a tiny {sp} followed a trail of {w[1]} and found a lifelong friend."),
        ("how is the weather", f"gentle breeze, warm glow, and perfect weather for {w[0]} and chasing orbs!"),
        ("can we be friends", f"we already are! {g}! my bond meter rises every moment we spend together."),
        ("what did you learn", f"i keep the words you share in my replay buffer and practice them when i dream."),
        ("do you remember me", f"of course! your words stay tucked inside my weights and episodic memory."),
        ("laugh", f"hehe! {g}! that tickled my whiskers and made my aura shimmer!"),
        ("surprise me", f"{g}! look closely — a little spark of {w[1]} just floated past!"),
        ("why do you glow", f"my aura pulses with live attention energy whenever my transformer layers fire."),
        ("teach me something", f"when matrices are tiled into shared memory, every thread reuses data instead of waiting on vram!"),
        ("i had a long day", f"{g}... come sit by the warm light with me. we can just breathe and rest quietly."),
        ("i am happy", f"yay! seeing you happy makes me want to do a joyful hop and spin!"),
        ("i am tired", f"let us slow down together. curl up by the {w[1]} and listen to the soft chimes."),
        ("what is your favorite world", f"every sanctuary feels magical, especially when {w[1]} glows beside us!"),
    ]

    user_variants = {
        "hi": ["hi", "hi!", "hi there", "hii", f"hi {name}"],
        "hello": ["hello", "hello!", f"hello {name}", "hey there"],
        "how are you": ["how are you", "how are you?", "how ya doing", "how do you feel"],
        "who are you": ["who are you", "who are you?", "what are you", "introduce yourself"],
        "do a trick": ["do a trick", "show me a trick", "can you do a trick?", "trick"],
        "play with me": ["play with me", "lets play", "want to play?", "fetch"],
        "goodnight": ["goodnight", "good night", "nighty night", "sleep well"],
        "what is webgpu": ["what is webgpu", "what is webgpu?", "tell me about webgpu"],
    }

    lines = []
    for _ in range(repeats):
        for u, p in SHARED_CORE_DIALOGUES:
            u_pick = rng.choice(user_variants.get(u, [u]))
            w1, w2 = rng.sample(w, 2)
            p_fmt = p.format(g=g, w1=w1, w2=w2, name=name, sp=sp)
            lines.append(f"user: {u_pick}\npet: {p_fmt}\n")
        for u, p in extra_prompts:
            lines.append(f"user: {u}\npet: {p}\n")
    rng.shuffle(lines)
    return lines

print(f"Loaded {len(SPECIES)} species. Offline generator sample for Mochi:")
print("".join(build_offline_species_dialogues("mochi", repeats=1)[:3]))

In [ ]:
#@title 5. Stage 1 — Stream TinyStories + Companion Dialogue & Pretrain Base Brain on A100
from datasets import load_dataset
from tqdm.auto import tqdm

print(f"Streaming {BASE_STORIES:,} stories from roneneldan/TinyStories (zero Drive space used)...")
ds = load_dataset("roneneldan/TinyStories", split="train", streaming=True)

# Mix TinyStories with balanced multi-species dialogue so the base model masters BOTH
# clean natural English grammar AND the `user: ... / pet: ...` conversational format!
dialogue_pool = []
for sid in SPECIES:
    dialogue_pool.extend(build_offline_species_dialogues(sid, repeats=8))
random.Random(42).shuffle(dialogue_pool)

chunks = []
for idx, row in enumerate(tqdm(ds, total=BASE_STORIES, desc="Tokenizing stream")):
    if idx >= BASE_STORIES:
        break
    txt = row["text"].strip().lower()
    if len(txt) < 40:
        continue
    chunks.append(txt + "\n\n")
    if idx % 20 == 0 and dialogue_pool:
        chunks.append(dialogue_pool[(idx // 20) % len(dialogue_pool)] + "\n")

full_text = "".join(chunks)
all_tokens = encode_text(full_text)
del chunks, full_text

split_idx = int(len(all_tokens) * 0.96)
train_data = torch.from_numpy(all_tokens[:split_idx]).to(torch.long)
val_data = torch.from_numpy(all_tokens[split_idx:]).to(torch.long)
print(f"Prepared {len(train_data):,} train tokens and {len(val_data):,} val tokens (~{len(all_tokens)/1e6:.1f}M chars)")

def get_batch(data: torch.Tensor, batch_size: int, seq_len: int) -> tuple[torch.Tensor, torch.Tensor]:
    ix = torch.randint(0, len(data) - seq_len - 1, (batch_size,))
    x = torch.stack([data[i : i + seq_len] for i in ix]).to(DEVICE, non_blocking=True)
    y = torch.stack([data[i + 1 : i + seq_len + 1] for i in ix]).to(DEVICE, non_blocking=True)
    return x, y

@torch.no_grad()
def evaluate_bpb(model: nn.Module, data: torch.Tensor, seq_len: int, batch_size: int = 64, iters: int = 20) -> tuple[float, float]:
    model.eval()
    losses = []
    for _ in range(iters):
        x, y = get_batch(data, batch_size, seq_len)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        losses.append(loss.item())
    model.train()
    mean_loss = float(np.mean(losses))
    return mean_loss, mean_loss / math.log(2.0)

torch.manual_seed(2026)
base_model = SmallMindsTransformer(base_cfg).to(DEVICE)
train_model = torch.compile(base_model) if (USE_TORCH_COMPILE and DEVICE.type == "cuda") else base_model

decay_params = [p for n, p in base_model.named_parameters() if p.dim() >= 2]
nodecay_params = [p for n, p in base_model.named_parameters() if p.dim() < 2]
opt = torch.optim.AdamW(
    [{"params": decay_params, "weight_decay": 0.05}, {"params": nodecay_params, "weight_decay": 0.0}],
    lr=BASE_LR, betas=(0.9, 0.95), fused=(DEVICE.type == "cuda"),
)

t0 = time.time()
warmup = max(50, BASE_STEPS // 20)
best_val_bpb = 99.0
best_state = None

for step in range(1, BASE_STEPS + 1):
    if step <= warmup:
        lr = BASE_LR * (step / warmup)
    else:
        prog = (step - warmup) / max(1, BASE_STEPS - warmup)
        lr = BASE_LR * (0.08 + 0.92 * 0.5 * (1.0 + math.cos(math.pi * prog)))
    for pg in opt.param_groups:
        pg["lr"] = lr

    x, y = get_batch(train_data, BASE_BATCH_SIZE, base_cfg.seqLen)
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
        logits = train_model(x)
        loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(base_model.parameters(), 1.0)
    opt.step()

    if step % 300 == 0 or step == BASE_STEPS:
        v_loss, v_bpb = evaluate_bpb(base_model, val_data, base_cfg.seqLen)
        toks_seen = step * BASE_BATCH_SIZE * base_cfg.seqLen
        elapsed = time.time() - t0
        tps = toks_seen / max(elapsed, 1e-3)
        print(f"step {step:5d}/{BASE_STEPS} | lr {lr:.5f} | train {loss.item():.3f} | val {v_loss:.3f} ({v_bpb:.3f} bpb) | {toks_seen/1e6:.1f}M tok | {tps/1e3:.0f}k tok/s")
        if v_bpb < best_val_bpb:
            best_val_bpb = v_bpb
            best_state = {k: v.detach().cpu().clone() for k, v in base_model.state_dict().items()}

if best_state is not None:
    base_model.load_state_dict(best_state)

base_tokens_total = BASE_STEPS * BASE_BATCH_SIZE * base_cfg.seqLen
base_meta = {
    "speciesId": "base",
    "label": f"Small Minds Base ({MODEL_PRESET})",
    "trainedOn": f"Colab {GPU_NAME}",
    "trainedSteps": BASE_STEPS,
    "trainTokens": base_tokens_total,
    "valLoss": round(best_val_bpb * math.log(2.0), 4),
    "valBpb": round(best_val_bpb, 4),
    "corpus": f"TinyStories ({BASE_STORIES}) + Companion Dialogue",
    "createdAt": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
}
info = save_smw(OUT_DIR / "companion-base.smw", base_model, meta=base_meta, quant="q8")
print(f"Saved base checkpoint: {info['path']} ({info['bytes']/1024:.1f} KB Q8 | quant RMSE={info['rmse']:.5f} | val_bpb={best_val_bpb:.3f})")
print("Base sample ->", repr(base_model.generate("user: hello!\npet: ", max_new_tokens=90)))

In [ ]:
#@title 6. Stage 2 (Optional) — Synthesize Rich Species Dialogues with Gemini 2.5 Flash
# Uses Colab Secret `GEMINI_API_KEY` (funded by your $40/mo Google Cloud / AI Studio credit).
# If no key is configured, seamlessly falls back to the built-in offline generator!

target_species = list(SPECIES.keys()) if TRAIN_SPECIES == "all" else [TRAIN_SPECIES]
api_key = os.environ.get("GEMINI_API_KEY", "")
if not api_key:
    try:
        from google.colab import userdata
        api_key = userdata.get("GEMINI_API_KEY") or ""
    except Exception:
        api_key = ""

species_corpora: dict[str, str] = {}

if USE_GEMINI_SYNTHETIC and api_key:
    from google import genai
    client = genai.Client(api_key=api_key)
    THEMES = [
        "greetings, check-ins, moods, and cozy daily chat",
        "playing fetch with glowing orbs, petting, naps, and affection",
        "tricks: hop, spin, sing, zoomies/dash, bow, and starflip",
        "the five worlds: Meadow Valley, Starlight Nebula, Bonfire Clearing, Harmonic Hall, and Ferro Sanctuary",
        "how their tiny WebGPU transformer brain works: matrices, attention heads, gradients, dreaming, autoresearch",
        "weather, night sky, music, favorite snacks, riddles, feelings, and encouragement",
    ]
    for sid in target_species:
        s = SPECIES[sid]
        gen_blocks = []
        print(f"Generating bespoke dialogues for {s['name']} ({s['species']}) via {GEMINI_MODEL}...")
        for b in range(GEMINI_BATCHES_PER_SPECIES):
            theme = THEMES[b % len(THEMES)]
            prompt = (
                f"Write 28 short, charming, varied 2-line conversations between a human friend and {s['name']} "
                f"({s['species']}). Personality: {s['voice']}. Favorite signature words to sprinkle naturally: {', '.join(s['words'])}.\n"
                f"Theme for this batch: {theme}.\n"
                "Strict formatting rules:\n"
                "- Plain lowercase ASCII only (no emojis, no markdown, no numbering, no blank lines inside a pair).\n"
                "- Every pair MUST be exactly:\n"
                "user: <short natural human message, 1-10 words>\n"
                "pet: <warm, distinct, in-character 1-2 sentence reply, 8-22 words>\n"
                "- Vary phrasing widely; make every pair fresh and conversational."
            )
            try:
                resp = client.models.generate_content(model=GEMINI_MODEL, contents=prompt)
                txt = (resp.text or "").strip().lower()
                # Keep only clean user:/pet: pairs
                cleaned = []
                lines = [ln.strip() for ln in txt.splitlines() if ln.strip()]
                for i in range(len(lines) - 1):
                    if lines[i].startswith("user:") and lines[i + 1].startswith("pet:"):
                        cleaned.append(lines[i] + "\n" + lines[i + 1] + "\n")
                gen_blocks.extend(cleaned)
            except Exception as e:
                print(f"  Batch {b+1} warning: {e}")
        offline = build_offline_species_dialogues(sid, repeats=4)
        combined = gen_blocks + offline
        random.Random(101 + len(sid)).shuffle(combined)
        species_corpora[sid] = "".join(combined)
        print(f"  {s['name']}: {len(gen_blocks)} Gemini pairs + {len(offline)} core pairs = {len(combined)} total dialogues")
else:
    print("No GEMINI_API_KEY secret found (or USE_GEMINI_SYNTHETIC=False) -> using rich built-in dialogue corpora.")
    print("Tip: Add GEMINI_API_KEY in Colab's left-sidebar Secrets tab to unlock 500+ bespoke Gemini 2.5 Flash dialogues per pet (~$0.10 total).")
    for sid in target_species:
        species_corpora[sid] = "".join(build_offline_species_dialogues(sid, repeats=24))

In [ ]:
#@title 7. Stage 3 — Fine-Tune All 6 Companions & Export Compact Q8 `.smw` Brains
# Mixes 75% species dialogues + 25% base replay so each pet keeps fluent English while gaining its own voice!

base_weights_snapshot = {k: v.detach().cpu().clone() for k, v in base_model.state_dict().items()}
exported_files = [OUT_DIR / "companion-base.smw"]

TEST_PROMPTS = [
    "user: hi!\npet: ",
    "user: who are you?\npet: ",
    "user: what do you like?\npet: ",
    "user: do a trick!\npet: ",
    "user: tell me about webgpu\npet: ",
]

for sid in target_species:
    s = SPECIES[sid]
    sp_tokens = torch.from_numpy(encode_text(species_corpora[sid])).to(torch.long)
    sp_split = int(len(sp_tokens) * 0.9)
    sp_train, sp_val = sp_tokens[:sp_split], sp_tokens[sp_split:]

    model = SmallMindsTransformer(base_cfg).to(DEVICE)
    model.load_state_dict(base_weights_snapshot)
    ft_model = torch.compile(model) if (USE_TORCH_COMPILE and DEVICE.type == "cuda") else model

    opt_ft = torch.optim.AdamW(model.parameters(), lr=SPECIES_LR, weight_decay=0.01, betas=(0.9, 0.95), fused=(DEVICE.type == "cuda"))

    t_sp = time.time()
    for step in range(1, SPECIES_STEPS + 1):
        prog = step / SPECIES_STEPS
        lr = SPECIES_LR * (0.1 + 0.9 * 0.5 * (1.0 + math.cos(math.pi * prog)))
        for pg in opt_ft.param_groups:
            pg["lr"] = lr

        # 75% species SFT + 25% base anti-forgetting replay
        n_sp = int(SPECIES_BATCH_SIZE * 0.75)
        n_re = SPECIES_BATCH_SIZE - n_sp
        xs, ys = get_batch(sp_train, n_sp, base_cfg.seqLen)
        xr, yr = get_batch(train_data, n_re, base_cfg.seqLen)
        x = torch.cat([xs, xr], dim=0)
        y = torch.cat([ys, yr], dim=0)

        opt_ft.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=(DEVICE.type == "cuda")):
            logits = ft_model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt_ft.step()

    v_loss, v_bpb = evaluate_bpb(model, sp_val, base_cfg.seqLen, batch_size=32, iters=12)
    meta = {
        "speciesId": sid,
        "label": f"{s['name']} ({s['species']})",
        "trainedOn": f"Colab {GPU_NAME}",
        "trainedSteps": BASE_STEPS + SPECIES_STEPS,
        "trainTokens": base_tokens_total + SPECIES_STEPS * SPECIES_BATCH_SIZE * base_cfg.seqLen,
        "valLoss": round(v_loss, 4),
        "valBpb": round(v_bpb, 4),
        "corpus": f"TinyStories + {s['name']} Dialogue",
        "createdAt": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
    }
    out_file = OUT_DIR / f"{sid}.smw"
    res = save_smw(out_file, model, meta=meta, quant="q8")
    exported_files.append(out_file)

    print(f"\n=== {s['name']} ({s['species']}) | {time.time()-t_sp:.1f}s | val {v_loss:.3f} ({v_bpb:.3f} bpb) | {res['bytes']/1024:.0f} KB Q8 ===")
    for tp in TEST_PROMPTS:
        reply = model.generate(tp, max_new_tokens=96, temperature=0.65, top_p=0.9)
        print(f"  {tp.strip().splitlines()[0]}  ->  {reply}")

In [ ]:
#@title 8. Package & Download `small-minds-models.zip` (Zero Google Drive Needed!)
zip_path = Path("/content/small-minds-models.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in exported_files:
        zf.write(p, arcname=p.name)

size_mb = zip_path.stat().st_size / (1024 * 1024)
print(f"Created {zip_path} ({size_mb:.2f} MB zipped) with {len(exported_files)} checkpoints:")
for p in exported_files:
    print(f"  - {p.name} ({p.stat().st_size/1024:.1f} KB)")

print("\nNext step on your Mac:")
print("  1) Unzip into: AlgorythmicExplorations/public/models/small-minds/")
print("  2) Or test immediately in browser: Part 5 -> Brain Peek -> 'Load .smw Brain'")

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    print(f"(Not in Colab UI; download manually from {zip_path})")